In [35]:
import pandas as pd
import requests
import re
import time

from bs4 import BeautifulSoup
from urllib.parse import quote_plus, urljoin
from pathlib import Path

In [36]:
df = pd.read_excel(r"D:\Proyek FEB\Publikasi internasional.xlsx")
df.head()

,No,Title,Authors,Journal,SCOPUS Q,Tahun,Volume / Issue,LINK DOI/ARTIKEL,SCOPUS SITASI
0,1,Ethnic identity and internal migration decisio...,ILMIAWAN AUWALIN,Journal of Ethnic and Migration Studies,?,2019-01-11 00:00:00,"Vol. 14, Issue 13",10.1080/1369183X.2018.1561252,?
1,2,The role of service quality within Indonesian ...,BADRI MUNIR SUKOCO,Journal of Islamic Marketing,?,2020-01-14 00:00:00,"Vol. 11, Issue 1",10.1108/JIMA-03-2017-0033,?
2,3,Developing Islamic crowdfunding website platfo...,MUHAMAD NAFIK HADI RYANDONO,Journal of Islamic Marketing,?,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,?
3,4,Developing Islamic crowdfunding website platfo...,ACHSANIA HENDRATMI,Journal of Islamic Marketing,?,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,?
4,5,Developing Islamic crowdfunding website platfo...,PUJI SUCIA SUKMANINGRUM,Journal of Islamic Marketing,?,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,?


In [37]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.strip()

In [38]:
def parse_citation(text):
    if not text:
        return None
    
    match = re.search(r"(\d+)\s+cited", text, flags=re.IGNORECASE)
    
    if match:
        return int(match.group(1))
    
    return None

In [39]:
def parse_quartile(text):
    if not text:
        return None
    
    match = re.search(r"\b(Q[1-4])\b", text, flags=re.IGNORECASE)
    
    if match:
        return match.group(1).upper()
    
    if re.search(r"\bno-?Q\b", text, flags=re.IGNORECASE):
        return "no-Q"
    
    return None

In [40]:
def is_link_filled(value):
    if pd.isna(value):
        return False
    
    value = str(value).strip()
    
    if value == "" or value.lower() in ["nan", "none", "-", "?"]:
        return False
    
    return True

In [41]:
def normalize_doi_or_link(value):
    if not value:
        return None
    
    value = str(value).strip()
    
    if value == "" or value.lower() in ["nan", "none", "-", "?"]:
        return None
    
    # Sudah URL
    if value.startswith("http://") or value.startswith("https://"):
        return value
    
    # DOI mentah
    if value.startswith("10."):
        return f"https://doi.org/{value}"
    
    return value

In [42]:
def extract_doi_from_soup(soup):
    if soup is None:
        return None
    
    # 1. Coba ambil dari data-testid publication-doi
    doi_el = soup.select_one('[data-testid="publication-doi"]')
    if doi_el:
        doi_text = doi_el.get_text(" ", strip=True)
        doi_text = doi_text.replace("DOI:", "").strip()
        if doi_text.startswith("10."):
            return doi_text
    
    # 2. Coba cari pola DOI di seluruh teks halaman
    text = soup.get_text(" ", strip=True)
    
    match = re.search(r"\b10\.\d{4,9}/[-._;()/:A-Z0-9]+\b", text, flags=re.IGNORECASE)
    
    if match:
        return match.group(0).strip()
    
    return None

In [43]:
def fetch_page(url, sleep_time=1.0):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        time.sleep(sleep_time)
        
        if response.status_code != 200:
            return None, response.status_code
        
        soup = BeautifulSoup(response.text, "html.parser")
        return soup, response.status_code
    
    except Exception:
        return None, None

In [44]:
def get_sinta_first_result(title, current_link=None, sleep_time=1.5):
    base_url = "https://sinta.kemdiktisaintek.go.id/scopus/"
    search_url = f"{base_url}?q={quote_plus(str(title))}"
    
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    
    try:
        response = requests.get(search_url, headers=headers, timeout=30)
        time.sleep(sleep_time)
        
        if response.status_code != 200:
            return {
                "title": title,
                "matched_title": None,
                "scopus_q": None,
                "scopus_citation": None,
                "final_link": current_link if is_link_filled(current_link) else None,
                "detail_or_scopus_url": None,
                "status": "request_failed",
                "http_status": response.status_code,
                "source_url": search_url
            }
        
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Ambil hasil pertama
        first_item = soup.select_one("div.ar-list-item")
        
        if first_item is None:
            return {
                "title": title,
                "matched_title": None,
                "scopus_q": None,
                "scopus_citation": None,
                "final_link": current_link if is_link_filled(current_link) else None,
                "detail_or_scopus_url": None,
                "status": "not_found",
                "http_status": response.status_code,
                "source_url": search_url
            }
        
        # Ambil elemen-elemen utama
        title_link_el = first_item.select_one(".ar-title a")
        title_el = first_item.select_one(".ar-title")
        cited_el = first_item.select_one(".ar-cited")
        quartile_el = first_item.select_one(".ar-quartile")
        year_el = first_item.select_one(".ar-year")
        
        matched_title = title_el.get_text(" ", strip=True) if title_el else None
        
        citation_text = cited_el.get_text(" ", strip=True) if cited_el else None
        quartile_text = quartile_el.get_text(" ", strip=True) if quartile_el else None
        year_text = year_el.get_text(" ", strip=True) if year_el else None
        
        scopus_citation = parse_citation(citation_text)
        scopus_q = parse_quartile(quartile_text)
        
        detail_or_scopus_url = None
        
        if title_link_el and title_link_el.get("href"):
            detail_or_scopus_url = urljoin(search_url, title_link_el.get("href"))
        
        # Logika link DOI/artikel
        if is_link_filled(current_link):
            final_link = normalize_doi_or_link(current_link)
            link_status = "existing_link_kept"
        else:
            final_link = None
            link_status = "link_missing"
            
            # Jika link judul langsung ke Scopus, simpan dulu sebagai fallback
            if detail_or_scopus_url:
                final_link = detail_or_scopus_url
                link_status = "article_link_from_sinta_result"
            
            # Kalau link detail bukan Scopus atau masih bisa dibuka, coba cari DOI di halaman tersebut
            if detail_or_scopus_url and "scopus.com" not in detail_or_scopus_url:
                detail_soup, detail_status = fetch_page(detail_or_scopus_url, sleep_time=1.0)
                doi = extract_doi_from_soup(detail_soup)
                
                if doi:
                    final_link = normalize_doi_or_link(doi)
                    link_status = "doi_found_from_detail"
        
        return {
            "title": title,
            "matched_title": matched_title,
            "sinta_year": year_text,
            "scopus_q": scopus_q,
            "quartile_text": quartile_text,
            "scopus_citation": scopus_citation,
            "citation_text": citation_text,
            "final_link": final_link,
            "detail_or_scopus_url": detail_or_scopus_url,
            "status": "found",
            "link_status": link_status,
            "http_status": response.status_code,
            "source_url": search_url
        }
    
    except Exception as e:
        return {
            "title": title,
            "matched_title": None,
            "scopus_q": None,
            "scopus_citation": None,
            "final_link": current_link if is_link_filled(current_link) else None,
            "detail_or_scopus_url": None,
            "status": "error",
            "error": str(e),
            "source_url": search_url
        }

# testing

In [45]:
sample_title = "Ethnic identity and internal migration decision in Indonesia"

result = get_sinta_first_result(sample_title)

result

{'title': 'Ethnic identity and internal migration decision in Indonesia',
 'matched_title': 'Ethnic identity and internal migration decision in Indonesia',
 'sinta_year': '2020',
 'scopus_q': 'Q1',
 'quartile_text': 'Q1 Journal',
 'scopus_citation': 24,
 'citation_text': '24 cited',
 'final_link': 'https://www.scopus.com/record/display.uri?eid=2-s2.0-85060010639&origin=resultslist',
 'detail_or_scopus_url': 'https://www.scopus.com/record/display.uri?eid=2-s2.0-85060010639&origin=resultslist',
 'status': 'found',
 'link_status': 'article_link_from_sinta_result',
 'http_status': 200,
 'source_url': 'https://sinta.kemdiktisaintek.go.id/scopus/?q=Ethnic+identity+and+internal+migration+decision+in+Indonesia'}

In [46]:
print("Title:", result["title"])
print("Matched:", result["matched_title"])
print("Q:", result["scopus_q"])
print("Citation:", result["scopus_citation"])
print("Final link:", result["final_link"])
print("Detail/Scopus URL:", result["detail_or_scopus_url"])
print("Status:", result["status"])

Title: Ethnic identity and internal migration decision in Indonesia
Matched: Ethnic identity and internal migration decision in Indonesia
Q: Q1
Citation: 24
Final link: https://www.scopus.com/record/display.uri?eid=2-s2.0-85060010639&origin=resultslist
Detail/Scopus URL: https://www.scopus.com/record/display.uri?eid=2-s2.0-85060010639&origin=resultslist
Status: found


In [47]:
df["title_normalized"] = df["Title"].apply(normalize_text)

unique_articles = (
    df[["Title", "title_normalized", "LINK DOI/ARTIKEL"]]
    .dropna(subset=["Title"])
    .drop_duplicates(subset=["title_normalized"])
    .reset_index(drop=True)
)

len(unique_articles)

1332

In [48]:
results = []

for i, row in unique_articles.iterrows():
    title = row["Title"]
    current_link = row["LINK DOI/ARTIKEL"]
    
    print(f"{i+1}/{len(unique_articles)} - {title[:90]}")
    
    result = get_sinta_first_result(
        title=title,
        current_link=current_link,
        sleep_time=1.5
    )
    
    results.append(result)

sinta_results_df = pd.DataFrame(results)

1/1332 - Ethnic identity and internal migration decision in Indonesia
2/1332 - The role of service quality within Indonesian customers satisfaction and loyalty and its i
3/1332 - Developing Islamic crowdfunding website platform for startup companies in Indonesia
4/1332 - Exploration of pilgrimage tourism in Indonesia
5/1332 - A critical assessment of retail sovereign sukuk in Indonesia
6/1332 - The impact of auditors’ awareness of the profession’s reputation for independence on audit
7/1332 - Exploring the role of visual aesthetics and presentation modality in luxury fashion brand 
8/1332 - Anger punishes, compassion forgives: How discrete emotions mitigate double standards in co
9/1332 - Managing paradoxes of innovation in an Indonesian TV group
10/1332 - Competitiveness and cost behaviour: evidence from the retail industry
11/1332 - Effects of Halal social media and customer engagement on brand satisfaction of Muslim cust
12/1332 - Islamic microfinance institution: Survey data from I

In [51]:
df1=pd.DataFrame(results)
df1

,title,matched_title,sinta_year,scopus_q,quartile_text,scopus_citation,citation_text,final_link,detail_or_scopus_url,status,link_status,http_status,source_url,error
0,Ethnic identity and internal migration decisio...,Ethnic identity and internal migration decisio...,2020,Q1,Q1 Journal,24.0,24 cited,https://doi.org/10.1080/1369183X.2018.1561252,https://www.scopus.com/record/display.uri?eid=...,found,existing_link_kept,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
1,The role of service quality within Indonesian ...,The role of service quality within Indonesian ...,2020,Q2,Q2 Journal,60.0,60 cited,https://doi.org/10.1108/JIMA-03-2017-0033,https://www.scopus.com/record/display.uri?eid=...,found,existing_link_kept,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
2,Developing Islamic crowdfunding website platfo...,Developing Islamic crowdfunding website platfo...,2020,Q2,Q2 Journal,48.0,48 cited,https://doi.org/10.1108/JIMA-02-2019-0022,https://www.scopus.com/record/display.uri?eid=...,found,existing_link_kept,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
3,Exploration of pilgrimage tourism in Indonesia,Exploration of pilgrimage tourism in Indonesia,2020,Q2,Q2 Journal,33.0,33 cited,https://doi.org/10.1108/JIMA-10-2018-0188,https://www.scopus.com/record/display.uri?eid=...,found,existing_link_kept,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
4,A critical assessment of retail sovereign suku...,A critical assessment of retail sovereign suku...,2020,Q2,Q2 Journal,9.0,9 cited,https://doi.org/10.1108/QRFM-10-2018-0109,https://www.scopus.com/record/display.uri?eid=...,found,existing_link_kept,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1327,Does economic uncertainty hinder or help busin...,None,NaN,None,NaN,NaN,NaN,10.1108/IMEFM-05-2024-0238,None,not_found,NaN,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
1328,Assessing the impact of economic uncertainty o...,None,NaN,None,NaN,NaN,NaN,10.1108/JIABR-05-2024-0175,None,not_found,NaN,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
1329,Do high-pollution firm need more environmental...,None,NaN,None,NaN,NaN,NaN,10.1108/SRJ-01-2025-0015,None,not_found,NaN,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
1330,"The impact of digitalization, education, and i...",None,NaN,None,NaN,NaN,NaN,10.1016/j.ssaho.2025.101423,None,not_found,NaN,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN


In [52]:
df1.to_excel(r"D:\Proyek FEB\lengkapin data\sinta_results.xlsx", index=False)